# Onboarding a new sensor export

The whole pipeline from a raw CSV to the master activity table. Run top to bottom.

See `onboarding.md` for the written procedure, including how to discover the label names
that go into `sensor_config.py` for an export this notebook has not seen before.

## 1. Load

`load_sensor_csv` names the columns, applies the label corrections, and builds a combined
`datetime` column. The first column is called `signal` rather than `location`, since it
holds a sensor state transition rather than a place.

In [1]:
import pandas as pd

import sensor_tools as st
import sensor_config as cfg

CSV_PATH = "HomeIntellex-2 - Sheet1.csv"

df_sorted = st.load_sensor_csv(
    CSV_PATH,
    sensor_type_fixes=cfg.SENSOR_TYPE_FIXES,
    signal_fixes=cfg.SIGNAL_TYPE_FIXES,
)
df_sorted.head()

[load_sensor_csv] n_in=- n_out=5319 65.40 ms


,signal,date,time,sensor_type,datetime
0,Left_Bedroom,09/05/2026,19:05:10,Onvis_MotionSensor,2026-09-05 19:05:10
1,In_Bedroom,09/05/2026,19:05:17,Onvis_MotionSensor,2026-09-05 19:05:17
2,Left_Bedroom,09/05/2026,19:05:27,Onvis_MotionSensor,2026-09-05 19:05:27
3,In_Bedroom,09/05/2026,19:05:28,Onvis_MotionSensor,2026-09-05 19:05:28
4,BedroomDoor,09/05/2026,19:05:36,Presence_FP2,2026-09-05 19:05:36


## 2. Verify the config covers the export

Any signal in the data that no config entry claims would silently vanish from the master
table, so check for that before trusting anything downstream.

In [2]:
mapped = set()
for states, events in cfg.SENSOR_FAMILIES:
    for labels in (states or {}).values():
        mapped.update(labels.values())
    mapped.update((events or {}).values())

unmapped = sorted(set(df_sorted["signal"].unique()) - mapped)
print(f"signals in export : {df_sorted['signal'].nunique()}")
print(f"claimed by config : {len(mapped)}")
print(f"unmapped          : {unmapped if unmapped else 'none'}")

signals in export : 34
claimed by config : 34
unmapped          : none


## 3. Master timeline

One row per signal, one column per sensor. A state column holds 1 from its on signal until
its off signal; a momentary column is 1 only on its own row. Every other column carries its
last value forward, so each row is the whole house at that instant.

`presence_zone` is the categorical view of the presence one-hot columns, kept alongside them
rather than replacing them.

In [3]:
master_timeline = st.build_master_timeline(
    df_sorted,
    cfg.SENSOR_FAMILIES,
    cfg.LABEL_COLUMNS,
)

print(f"{master_timeline.shape[0]} rows x {master_timeline.shape[1]} columns")
master_timeline.head(15)

[build_master_timeline] n_in=5319 n_out=5319 277.32 ms
5319 rows x 21 columns


,datetime,signal,vib_bike,vib_bedroom_computer,vib_office_computer,vib_microwave,motion_kitchen,motion_office,motion_bedroom,contact_fridge_door,...,presence_sofa,presence_tableside,presence_chair,presence_tvzone,presence_bikeside,presence_backyard,presence_livingroom,presence_livingroom_inout,presence_bedroom_door,presence_zone
0,2026-09-05 19:05:10,Left_Bedroom,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none
1,2026-09-05 19:05:17,In_Bedroom,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,none
2,2026-09-05 19:05:27,Left_Bedroom,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none
3,2026-09-05 19:05:28,In_Bedroom,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,none
4,2026-09-05 19:05:36,BedroomDoor,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,none
5,2026-09-05 19:05:38,TVzone-In,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,presence_tvzone
6,2026-09-05 19:05:39,TVzone-Out,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,none
7,2026-09-05 19:05:41,Sofa-Sit,0,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,presence_sofa
8,2026-09-05 19:05:45,Left_Bedroom,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,presence_sofa
9,2026-09-05 19:06:02,Sofa-Left,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none


In [4]:
# Per-sensor totals. Momentary columns report a count only: they stay 1 until the next
# signal from any sensor, so their duration would measure that gap, not a real span.
master_summary = st.summarise_timeline(
    master_timeline,
    event_columns=cfg.PRESENCE_EVENT_COLUMNS,
)
display(master_summary.style)

[summarise_timeline] n_in=5319 n_out=18 23.45 ms


,kind,activations,time_active,pct_of_span
sensor,,,,
vib_bike,state,10,0 days 00:33:43,0.430000
vib_bedroom_computer,state,50,0 days 00:49:37,0.630000
vib_office_computer,state,628,0 days 11:12:39,8.490000
vib_microwave,state,20,0 days 00:23:02,0.290000
motion_kitchen,state,376,0 days 04:07:07,3.120000
motion_office,state,203,0 days 01:13:50,0.930000
motion_bedroom,state,70,0 days 00:17:57,0.230000
contact_fridge_door,state,53,0 days 00:24:42,0.310000
contact_garage_entrance,state,15,0 days 00:02:54,0.040000


## 4. Per-sensor activity tables

The master timeline is raw state. `build_activities` is the other view: it applies the
2 minute chaining rule, so nearby on/off cycles merge into one activity. The two disagree
by design, and the gap limit is the knob:

    bike, gap_limit=0min  -> 12 activities
    bike, gap_limit=2min  ->  6 activities
    bike, gap_limit=60min ->  5 activities

In [5]:
activities = {}

for family, labels_by_name in (
    ("vib", cfg.VIBRATION_SENSORS),
    ("motion", cfg.MOTION_SENSORS),
):
    for name, labels in labels_by_name.items():
        sub = st.subset_by_signal(df_sorted, [labels["on"], labels["off"]], telemetry=False)
        activities[f"{family}_{name}"] = st.build_activities(
            sub, labels["on"], labels["off"], telemetry=False
        )

# contact sensors alternate cleanly, so they are paired directly with no gap rule
for name, labels in cfg.CONTACT_SENSORS.items():
    sub = st.subset_by_signal(df_sorted, [labels["on"], labels["off"]], telemetry=False)
    activities[f"contact_{name}"] = st.build_paired_activities(
        sub, labels["on"], labels["off"], telemetry=False
    )

pd.DataFrame(
    [
        {"sensor": name, "activities": len(a), "total_time": a["duration"].sum()}
        for name, a in activities.items()
    ]
).set_index("sensor")

,activities,total_time
sensor,,
vib_bike,6,0 days 00:39:06
vib_bedroom_computer,36,0 days 01:02:22
vib_office_computer,284,0 days 15:47:22
vib_microwave,15,0 days 00:30:16
motion_kitchen,136,0 days 05:36:58
motion_office,116,0 days 02:04:29
motion_bedroom,33,0 days 00:36:19
contact_fridge_door,53,0 days 00:24:42
contact_garage_entrance,15,0 days 00:02:54


### Individual tables for the data viewer

The dict above is convenient for building and comparing, but Data Wrangler can only open a
named DataFrame. These bindings exist so each table can be opened directly.

In [6]:
bike_activities = activities["vib_bike"]
bedroom_computer_activities = activities["vib_bedroom_computer"]
office_computer_activities = activities["vib_office_computer"]
microwave_activities = activities["vib_microwave"]

kitchen_activities = activities["motion_kitchen"]
office_motion_activities = activities["motion_office"]
bedroom_motion_activities = activities["motion_bedroom"]

fridge_activities = activities["contact_fridge_door"]
garage_activities = activities["contact_garage_entrance"]

bike_activities

,start,duration,end
0,2026-09-05 21:47:29,0 days 00:01:37,2026-09-05 21:49:06
1,2026-09-06 11:26:18,0 days 00:05:17,2026-09-06 11:31:35
2,2026-09-06 13:54:41,0 days 00:02:22,2026-09-06 13:57:03
3,2026-09-06 14:11:23,0 days 00:01:00,2026-09-06 14:12:23
4,2026-09-07 08:34:52,0 days 00:26:54,2026-09-07 09:01:46
5,2026-09-11 05:49:21,0 days 00:01:56,2026-09-11 05:51:17


## 5. Display variant

Same master table with 1 rendered as `####1`, so an active sensor stands out against the
zeros. Display only: these columns are strings, so keep using `master_timeline` for
anything numeric.

In [7]:
state_cols = [
    c for c in master_timeline.columns
    if c not in ("datetime", "signal") and master_timeline[c].dtype.kind in "iu"
]

master_timeline_display = master_timeline.copy()
master_timeline_display[state_cols] = master_timeline[state_cols].replace({1: "####1", 0: "0"})

master_timeline_display.head(10)

,datetime,signal,vib_bike,vib_bedroom_computer,vib_office_computer,vib_microwave,motion_kitchen,motion_office,motion_bedroom,contact_fridge_door,...,presence_sofa,presence_tableside,presence_chair,presence_tvzone,presence_bikeside,presence_backyard,presence_livingroom,presence_livingroom_inout,presence_bedroom_door,presence_zone
0,2026-09-05 19:05:10,Left_Bedroom,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none
1,2026-09-05 19:05:17,In_Bedroom,0,0,0,0,0,0,####1,0,...,0,0,0,0,0,0,0,0,0,none
2,2026-09-05 19:05:27,Left_Bedroom,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none
3,2026-09-05 19:05:28,In_Bedroom,0,0,0,0,0,0,####1,0,...,0,0,0,0,0,0,0,0,0,none
4,2026-09-05 19:05:36,BedroomDoor,0,0,0,0,0,0,####1,0,...,0,0,0,0,0,0,0,0,####1,none
5,2026-09-05 19:05:38,TVzone-In,0,0,0,0,0,0,####1,0,...,0,0,0,####1,0,0,0,0,0,presence_tvzone
6,2026-09-05 19:05:39,TVzone-Out,0,0,0,0,0,0,####1,0,...,0,0,0,0,0,0,0,0,0,none
7,2026-09-05 19:05:41,Sofa-Sit,0,0,0,0,0,0,####1,0,...,####1,0,0,0,0,0,0,0,0,presence_sofa
8,2026-09-05 19:05:45,Left_Bedroom,0,0,0,0,0,0,0,0,...,####1,0,0,0,0,0,0,0,0,presence_sofa
9,2026-09-05 19:06:02,Sofa-Left,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,none


## 6. Data quality

Where a sensor repeats an on signal with no off in between. Contact sensors should report
zero; the presence and vibration sensors do not, which is why the activity builders have a
duplicate-handling strategy at all.

In [8]:
checks = []
for family, labels_by_name in (
    ("vib", cfg.VIBRATION_SENSORS),
    ("motion", cfg.MOTION_SENSORS),
    ("contact", cfg.CONTACT_SENSORS),
    ("presence", cfg.PRESENCE_STATES),
):
    for name, labels in labels_by_name.items():
        sub = st.subset_by_signal(df_sorted, [labels["on"], labels["off"]], telemetry=False)
        problems = st.check_alternation(sub, labels["on"], labels["off"], telemetry=False)
        checks.append({
            "sensor": f"{family}_{name}",
            "rows": len(sub),
            "violations": len(problems),
        })

display(pd.DataFrame(checks).set_index("sensor").sort_values("violations", ascending=False).style)

,rows,violations
sensor,,
presence_sofa,1221,35
presence_tableside,484,12
motion_kitchen,757,9
vib_office_computer,1260,6
motion_bedroom,141,3
vib_bedroom_computer,102,2
motion_office,408,2
vib_bike,22,2
presence_chair,100,2


## 7. Gap limit sensitivity

The gap limit decides when two nearby on/off cycles are one activity or two, and there is
no universally right value. Widening it can only merge activities, never split them, so
each row below is non-increasing.

Read each row for a **plateau**. A count that stops changing means the threshold has
stopped mattering, and that flat region is the defensible choice. A count that keeps
falling all the way across has no natural session boundary in the data, so the value has
to come from somewhere else — usually by asking the participant how often they actually
do the thing.

In [9]:
state_sensors = {**cfg.VIBRATION_STATES, **cfg.MOTION_STATES}

gap_sweep = st.sweep_all_sensors(
    df_sorted,
    state_sensors,
    cfg.GAP_LIMIT_SWEEP_MINUTES,
)
display(gap_sweep.style)

[sweep_all_sensors] n_in=5319 n_out=7 199.00 ms


gap_limit_min,0.000000,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000,15.000000
sensor,,,,,,,,,,
vib_bike,12,7,6,6,6,6,6,6,6,6
vib_bedroom_computer,52,40,36,35,33,33,32,29,27,22
vib_office_computer,630,399,284,222,170,131,110,92,75,35
vib_microwave,20,19,15,15,15,15,15,15,15,14
motion_kitchen,375,160,136,123,114,109,106,102,95,77
motion_office,203,133,116,100,95,93,89,87,85,68
motion_bedroom,71,38,33,30,22,18,18,17,13,10


In [10]:
# One sensor in detail, with total and median duration alongside the count.
bike_sub = st.subset_by_signal(df_sorted, ["Bike-InUse", "Bike-Stopped"], telemetry=False)

bike_gap_sweep = st.gap_limit_sweep(
    bike_sub, "Bike-InUse", "Bike-Stopped", cfg.GAP_LIMIT_SWEEP_MINUTES,
)
bike_gap_sweep

[gap_limit_sweep] n_in=22 n_out=10 17.61 ms


,activities,total_time,median_duration
gap_limit_min,,,
0.0,12,0 days 00:35:43,0 days 00:01:57.500000
1.0,7,0 days 00:37:27,0 days 00:01:56
2.0,6,0 days 00:39:06,0 days 00:02:09
3.0,6,0 days 00:39:06,0 days 00:02:09
4.0,6,0 days 00:39:06,0 days 00:02:09
5.0,6,0 days 00:39:06,0 days 00:02:09
6.0,6,0 days 00:39:06,0 days 00:02:09
7.0,6,0 days 00:39:06,0 days 00:02:09
8.0,6,0 days 00:39:06,0 days 00:02:09


### Choosing a per-sensor gap limit

`cfg.GAP_LIMITS_MINUTES` holds one value per sensor, so a sensor with a clear plateau can
use it while others keep the default. Combined with a participant's own estimate — "I ride
about once a day" — the sweep turns into a way to pick the threshold rather than guess it:
find the gap limit whose count is closest to the reported rate.

In [11]:
reported_rides_per_day = 1  # what the participant says

span_days = (df_sorted["datetime"].max() - df_sorted["datetime"].min()).total_seconds() / 86400
target = reported_rides_per_day * span_days

fit = bike_gap_sweep.copy()
fit["expected"] = round(target, 1)
fit["error"] = (fit["activities"] - target).abs()
fit.sort_values("error").head()

,activities,total_time,median_duration,expected,error
gap_limit_min,,,,,
3.0,6,0 days 00:39:06,0 days 00:02:09,5.5,0.499595
2.0,6,0 days 00:39:06,0 days 00:02:09,5.5,0.499595
5.0,6,0 days 00:39:06,0 days 00:02:09,5.5,0.499595
4.0,6,0 days 00:39:06,0 days 00:02:09,5.5,0.499595
6.0,6,0 days 00:39:06,0 days 00:02:09,5.5,0.499595


## 8. Location estimate

A single best guess at which room the person is in, at every row.

This is **not** the presence sensor column. `presence_zone` reports what the FP2 claims;
this column weighs every sensor in the house and commits to one answer.

### Events, not states

A signal is a **timestamped observation** of where someone was at that instant, not a
state that stays true. The most recent observation always wins — strength never lets an
older signal outrank a newer one.

This matters because sensors report departures late. A motion sensor may not send its off
signal until a minute after the person left, by which time they have already triggered a
presence zone elsewhere. Weighing sensor *state* rather than signal *events* loses that
race and strands the person in the room they left.

So the priority list decides **whether a signal is an observation at all**, not which of
two signals wins:

- tiers `0..LOCATION_POSITIVE_TIERS-1` are **positive**: a person was here, now. They move
  the estimate.
- the remaining tiers are **negative**: a zone reporting itself vacated. They place nobody
  anywhere, so they only falsify, clearing the estimate to `unknown`.

### Why the presence sensor alone is not enough

The FP2 reports more than one zone at once for **28% of rows** here. Most is structural —
`presence_livingroom` is a superset overlapping sofa, chair, table and TV. Excluding it
leaves **257 genuinely ambiguous rows**, and in half of those the older zone had not been
refreshed for over 2 minutes, which is what makes "most recent wins" a sound tiebreak.

In [12]:
location_estimate = st.estimate_location(
    df_sorted,
    cfg.SIGNAL_LOCATIONS,
    cfg.SIGNAL_PRIORITY,
    positive_tiers=cfg.LOCATION_POSITIVE_TIERS,
    decay=pd.Timedelta(minutes=cfg.LOCATION_DECAY_MINUTES),
    zone_exit_grace=pd.Timedelta(seconds=cfg.LOCATION_ZONE_EXIT_GRACE_SECONDS),
)

print(f"{(location_estimate['location'] != location_estimate['location'].shift()).sum()} location changes")
display(location_estimate.head(20).style)

[estimate_location] n_in=5319 n_out=5319 59.24 ms
1967 location changes


,datetime,signal,location,evidence_rank,evidence_age
0,2026-09-05 19:05:10,Left_Bedroom,unknown,nan,NaT
1,2026-09-05 19:05:17,In_Bedroom,bedroom,0.000000,0 days 00:00:00
2,2026-09-05 19:05:27,Left_Bedroom,bedroom,0.000000,0 days 00:00:10
3,2026-09-05 19:05:28,In_Bedroom,bedroom,0.000000,0 days 00:00:00
4,2026-09-05 19:05:36,BedroomDoor,bedroom,1.000000,0 days 00:00:00
5,2026-09-05 19:05:38,TVzone-In,livingroom,1.000000,0 days 00:00:00
6,2026-09-05 19:05:39,TVzone-Out,livingroom,1.000000,0 days 00:00:01
7,2026-09-05 19:05:41,Sofa-Sit,livingroom,1.000000,0 days 00:00:00
8,2026-09-05 19:05:45,Left_Bedroom,livingroom,1.000000,0 days 00:00:04
9,2026-09-05 19:06:02,Sofa-Left,livingroom,1.000000,0 days 00:00:21


In [13]:
# Time spent per location, and which evidence tier was driving the estimate.
held = df_sorted["datetime"].diff().shift(-1)

location_totals = (
    pd.DataFrame({"location": location_estimate["location"], "held": held})
    .groupby("location")["held"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "time", "count": "rows"})
    .sort_values("time", ascending=False)
)
location_totals["pct"] = (
    location_totals["time"] / location_totals["time"].sum() * 100
).round(1)
location_totals

,time,rows,pct
location,,,
unknown,2 days 02:31:01,780,38.3
bedroom,1 days 18:24:47,495,32.1
garage,0 days 15:39:32,42,11.9
office,0 days 10:45:25,1097,8.1
livingroom,0 days 09:09:58,2101,6.9
kitchen,0 days 03:29:47,800,2.6
backyard,0 days 00:00:05,3,0.0


> **Read these totals with care.** Decay is evaluated when a row is processed, so a
> 30-minute decay cannot expire an estimate *during* an 11-hour silence — the next row that
> would trigger the check arrives only when the silence ends. The per-row estimate is
> correct; what is approximate is any duration obtained by forward-filling it. The garage's
> 11.9% is one such artifact: a single 11h20m stretch while the person was out for the day.
>
> For trustworthy durations, emit explicit `unknown` rows at decay boundaries or resample
> onto a fixed grid before integrating.

### The two windows

Negative signals need care at two different timescales.

**`LOCATION_ZONE_EXIT_GRACE_SECONDS`** handles the short end. Rooms hold several zones, so
a zone exit usually means the person crossed their own livingroom rather than left it.
Falsifying on every zone exit makes the estimate flicker: `Sofa-Left`, then `TableSide-In`
8 seconds later. Measured here, **1544 of 1680 vacancy-driven unknowns were resolved by
the same room**, at a median of 7 seconds.

**`LOCATION_DECAY_MINUTES`** handles the long end: without it the last signal before a
silent night holds that room until morning. Gaps over an hour are **50% of this export's
span**, and 9 of the 15 longest begin with `Livingroom-Vacant`.

The sweep below should show the change count falling sharply while the unknown share
barely moves — the signature of removing artifacts rather than suppressing real
transitions.

In [14]:
rows = []
for secs in (None, 5, 15, 30, 60, 120):
    est = st.estimate_location(
        df_sorted, cfg.SIGNAL_LOCATIONS, cfg.SIGNAL_PRIORITY,
        positive_tiers=cfg.LOCATION_POSITIVE_TIERS,
        decay=pd.Timedelta(minutes=cfg.LOCATION_DECAY_MINUTES),
        zone_exit_grace=None if secs is None else pd.Timedelta(seconds=secs),
        telemetry=False,
    )
    share = (
        pd.DataFrame({"location": est["location"], "held": held})
        .groupby("location")["held"].sum()
    )
    share = (share / share.sum() * 100).round(1)
    rows.append({
        "grace_sec": "off" if secs is None else secs,
        "changes": int((est["location"] != est["location"].shift()).sum()),
        **share.to_dict(),
    })

display(pd.DataFrame(rows).set_index("grace_sec").style)

,changes,backyard,bedroom,garage,kitchen,livingroom,office,unknown
grace_sec,,,,,,,,
off,4029,0.000000,32.100000,11.900000,2.400000,6.500000,7.700000,39.500000
5,2608,0.000000,32.100000,11.900000,2.500000,6.700000,7.700000,39.300000
15,2249,0.000000,32.100000,11.900000,2.500000,6.800000,7.800000,38.900000
30,1967,0.000000,32.100000,11.900000,2.600000,6.900000,8.100000,38.300000
60,1733,0.000000,32.200000,11.900000,2.800000,7.100000,9.000000,37.200000
120,1463,0.000000,32.300000,11.900000,2.800000,7.300000,11.000000,34.700000


### One-hot, without overlap

If the label column cannot be used — building tensors, for instance — these columns carry
the same information with exactly one 1 per row. Because they are derived from a single
committed estimate rather than from the raw zone signals, no two locations can be active
in the same time slot.

In [15]:
location_one_hot = st.one_hot_location(location_estimate)

assert (location_one_hot.sum(axis=1) <= 1).all(), "locations must be mutually exclusive"
print("max locations active in any row:", location_one_hot.sum(axis=1).max())

display(location_one_hot.head(20).style)

[one_hot_location] n_in=5319 n_out=5319 6.31 ms
max locations active in any row: 1


,at_backyard,at_bedroom,at_garage,at_kitchen,at_livingroom,at_office
0,0,0,0,0,0,0
1,0,1,0,0,0,0
2,0,1,0,0,0,0
3,0,1,0,0,0,0
4,0,1,0,0,0,0
5,0,0,0,0,1,0
6,0,0,0,0,1,0
7,0,0,0,0,1,0
8,0,0,0,0,1,0
9,0,0,0,0,1,0


### Joined onto the master timeline

The estimate shares the master timeline's row order, so it attaches directly. The result
carries the raw per-sensor state, the presence sensor's own claim, and the single
best-guess location side by side.

In [16]:
master_with_location = master_timeline.copy()
master_with_location["location"] = location_estimate["location"].values
master_with_location["location_rank"] = location_estimate["evidence_rank"].values

print(f"{master_with_location.shape[0]} rows x {master_with_location.shape[1]} columns")
display(
    master_with_location[
        ["datetime", "signal", "presence_zone", "location", "location_rank"]
    ]
    .head(25)
    .style.format({"location_rank": "{:.0f}"}, na_rep="-")
)

5319 rows x 23 columns


,datetime,signal,presence_zone,location,location_rank
0,2026-09-05 19:05:10,Left_Bedroom,none,unknown,-
1,2026-09-05 19:05:17,In_Bedroom,none,bedroom,0
2,2026-09-05 19:05:27,Left_Bedroom,none,bedroom,0
3,2026-09-05 19:05:28,In_Bedroom,none,bedroom,0
4,2026-09-05 19:05:36,BedroomDoor,none,bedroom,1
5,2026-09-05 19:05:38,TVzone-In,presence_tvzone,livingroom,1
6,2026-09-05 19:05:39,TVzone-Out,none,livingroom,1
7,2026-09-05 19:05:41,Sofa-Sit,presence_sofa,livingroom,1
8,2026-09-05 19:05:45,Left_Bedroom,presence_sofa,livingroom,1
9,2026-09-05 19:06:02,Sofa-Left,none,livingroom,1
